In [1]:
import pandas as pd

In [2]:
def load_student_dataset(path):
    df = pd.read_csv(path)

    # Data standardization -> process of converting data from sources into a common, consistent format for the purpose of analysis.

    class_level_map = {
        10: "SS1",
        11: "SS2",
        12: "SS3"
    }

    df["English Language"] = pd.to_numeric(df["English Language"], errors="coerce")
    df["Literature in English"] = pd.to_numeric(df["Literature in English"], errors="coerce")

    #  Replaces all values. Any value for the column that's not mapped in the dictionary key, is replace with NaN (meaning an empty value)
    # df["class_level"] = df["class_level"].map(class_level_map)

    # Only replaces matched values. Does not attempt to replace values not provided in the dictionary key.
    df["class_level"] = df["class_level"].replace(class_level_map)

    return df

### Students with Over 92% Attendance Rate

In [9]:
def filter_ss1_students_by_attendance_rate(df: pd.DataFrame, class_level = "SS1", attendance_rate = 92):
    """Returns a DataFrame of SS1 students with the minimum attendance rate, as provided"""
    results = df[
        (df["class_level"] == class_level) &
        (df["attendance"] > attendance_rate)
    ]

    return results[["student_id", "first_name", "last_name", "class_level", "attendance"]]

### Students with Over 85% in English Language and Literature in English

In [4]:
def find_students_excelling_in_humanities(df, score = 85):
    results = df[
        (df["English Language"] > score) &
        (df["Literature in English"] > score)
    ]

    # results = df.query("'English Language' > @score and 'Literature in English' > @score")

    return results[["student_id", "first_name", "last_name", "English Language", "Literature in English"]]

In [ ]:
# TODO: Reference for dynamic filter
def find_students_excelling_in_humanities_alt(df, subjects = ["English Language", "Literature in English"], score = 85):
    queries = []

    for subject in subjects:
        queries.append(df[subject] > score)
    
    results = df[queries[0] & queries[1]]

    return results[["student_id", "first_name", "last_name", "English Language", "Literature in English"]]

# OR- better in the case where subjects are more than one.
# def find_students_excelling_in_humanities_alt(
#     df,
#     subjects=["English Language", "Literature in English"],
#     score=85
# ):

#     results = df[(df[subjects] > score).all(axis=1)]

#     return results[["student_id", "first_name", "last_name"] + subjects]

In [6]:
import random

# random.seed(80)
# TODO: Explain random seed

def find_students_needing_academic_support(df: pd.DataFrame, stem_subjects = None, max_score = 70):
    if stem_subjects is None:
        stem_subjects = ["Mathematics", "Chemistry", "Further Mathematics", "Physics", "Computer Science", "Biology"]


    # Approach 1
    choice_subjects = random.choices(stem_subjects, k=2)

    queries = []

    for subject in choice_subjects:
        queries.append(df[subject] < max_score)

    students_needing_support = df[
        (df["study_group"] == "Science") &
        queries[0] & queries[1]
    ]

    result_a = students_needing_support[["student_id", "first_name", "class_level", *choice_subjects]]
    print(result_a)
    
    print("\n")
    print("=="*50)
    print("\n")

    # Approach 2
    science_students = df[df["study_group"] == "Science"]
    students = [] # Needing support

    for index, student in science_students.iterrows():
        low_subjects = []

        for subject in stem_subjects:
            subject_score = student[subject]

            if subject_score < max_score:
                low_subjects.append(subject)

        if len(low_subjects) >= 2:
            student_info = {
                "students_id": student["student_id"],
                "first_name": student["first_name"],
                "last_name": student["last_name"],
                "low_subjects": ", ".join(low_subjects),
                "low_subjects_count": len(low_subjects),
            }

            students.append(student_info)

    result = pd.DataFrame(students)

    return result

In [7]:
students_df = load_student_dataset("../../data/students.csv")

# ss1_students_by_attendance = filter_ss1_students_by_attendance_rate(students_df)

humanities_excellence = find_students_needing_academic_support(students_df)

print(humanities_excellence)

            student_id first_name class_level  Further Mathematics  \
4      ALA-20240801005      Nneka         SS1                 46.0   
12     ALA-20240801013     Gbenga         SS1                 66.0   
17     ALA-20240826018      Bunmi         SS1                 64.0   
20     ALA-20240723021  Oluwaseyi         SS1                 66.0   
30     ALA-20240910031     Samuel         SS1                 63.0   
...                ...        ...         ...                  ...   
2394  ALA-202309232395     Ikenna         SS3                 37.0   
2395  ALA-202206272396     Obinna         SS3                 44.0   
2396  ALA-202309232397     Gbenga         SS3                 63.0   
2397  ALA-202307252398      David         SS3                 41.0   
2398  ALA-202209162399    Abiodun         SS3                 63.0   

      Further Mathematics  
4                    46.0  
12                   66.0  
17                   64.0  
20                   66.0  
30                 